# Prompting vs Fine-Tuning for Text Classification

**Goal:** run a controlled experiment comparing three approaches to the same text classification task, measure them on the same test set, and interpret when each approach is the right choice.

---

## The Three Approaches

| Approach | What changes | What stays fixed |
|---|---|---|
| **Zero-shot prompting** | Nothing — the model classifies from instructions alone | Base model weights |
| **Few-shot prompting** | A handful of labeled examples are added to the prompt | Base model weights |
| **LoRA fine-tuning** | Adapter weights are trained on labeled data | Base model weights (frozen) |

## Why This Comparison Matters

Teams often jump to fine-tuning because it sounds more serious, or stay on prompting because it is easier. Neither instinct is reliable. This notebook runs both under fair conditions so you can see:

- where prompting already saturates
- where fine-tuning adds real lift
- what the cost and maintenance trade-offs look like

## Experimental Design

```
                    Same labeled dataset
                           │
              ┌────────────┼────────────┐
              │             │            │
         Zero-shot      Few-shot     Fine-tuned
         prompting      prompting    (LoRA)
              │             │            │
              └─────────────┼────────────┘
                           │
                    Same held-out test set
                           │
                    Same metrics (F1, accuracy, per-class breakdown)
```

### Key Experimental Rules

1. **Same test set** for all three — never leak test examples into few-shot prompts or training data
2. **Same label taxonomy** — identical class names and definitions everywhere
3. **Same evaluation code** — one scoring function applied to all outputs
4. **Document every choice** — model, temperature, prompt template, training hyperparameters

## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy scikit-learn datasets transformers peft trl accelerate langchain-ollama

In [ ]:
import json
import random
import time
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

RUN_PROMPTING = True         # Zero-shot and few-shot via Ollama
RUN_FINE_TUNING = False      # LoRA training (requires GPU + time)
RUN_FT_EVAL = False          # Fine-tuned model inference on test set

OLLAMA_MODEL = "qwen3"
OLLAMA_BASE_URL = "http://localhost:11434"
BASE_HF_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print(f"Project dir:       {PROJECT_DIR}")
print(f"Prompting model:   {OLLAMA_MODEL} (Ollama)")
print(f"Fine-tune base:    {BASE_HF_MODEL}")
print(f"Run prompting:     {RUN_PROMPTING}")
print(f"Run fine-tuning:   {RUN_FINE_TUNING}")
print(f"Run FT eval:       {RUN_FT_EVAL}")

## 2. Task Definition — Customer Support Intent Classification

### Why This Task?

Intent classification is a common real-world task where both prompting and fine-tuning are viable. It has clear right/wrong answers, multiple classes, and varying difficulty levels — some intents are easy to distinguish, others overlap.

### Label Taxonomy

| Label | Description | Example |
|---|---|---|
| `billing` | Payment, charges, invoices, refunds | "Why was I charged twice?" |
| `technical` | Bugs, errors, setup, integrations | "The app crashes when I open settings" |
| `account` | Login, password, profile, permissions | "I can't reset my password" |
| `shipping` | Delivery, tracking, returns | "Where is my package?" |
| `general` | Feedback, feature requests, other | "Can you add dark mode?" |

In [ ]:
LABELS = ["billing", "technical", "account", "shipping", "general"]

LABEL_DESCRIPTIONS = {
    "billing": "Payment issues, charges, invoices, refunds, subscription pricing",
    "technical": "Bugs, errors, app crashes, setup problems, integration issues",
    "account": "Login, password reset, profile changes, permissions, account access",
    "shipping": "Delivery status, tracking, returns, address changes, shipping delays",
    "general": "Feedback, feature requests, compliments, questions that do not fit other categories",
}

print("Label taxonomy:")
for label, desc in LABEL_DESCRIPTIONS.items():
    print(f"  {label:12s} — {desc}")

## 3. Build the Dataset

We use a synthetic dataset so the notebook is self-contained. Each example has:

- a customer message
- a gold label
- a difficulty tag (easy / ambiguous) for slice analysis

### Why Difficulty Tags?

Average accuracy hides where the model struggles. Ambiguous examples — messages that could plausibly belong to two categories — are the most informative for comparing approaches.

In [ ]:
examples = [
    # ── billing (easy) ──
    {"text": "Why was I charged $49.99 instead of $39.99 this month?", "label": "billing", "difficulty": "easy"},
    {"text": "I need a refund for the duplicate charge on my credit card.", "label": "billing", "difficulty": "easy"},
    {"text": "Can I switch from monthly to annual billing?", "label": "billing", "difficulty": "easy"},
    {"text": "My invoice from October is missing. Can you resend it?", "label": "billing", "difficulty": "easy"},
    {"text": "I cancelled last week but I was still charged today.", "label": "billing", "difficulty": "easy"},
    {"text": "What payment methods do you accept?", "label": "billing", "difficulty": "easy"},
    {"text": "The promo code SAVE20 is not applying the discount at checkout.", "label": "billing", "difficulty": "easy"},
    {"text": "How do I update the credit card on my account?", "label": "billing", "difficulty": "ambiguous"},
    {"text": "I see a pending charge but I haven't bought anything.", "label": "billing", "difficulty": "easy"},
    {"text": "Can I get an itemized receipt for my last purchase?", "label": "billing", "difficulty": "easy"},

    # ── technical (easy) ──
    {"text": "The app crashes every time I try to open the settings page.", "label": "technical", "difficulty": "easy"},
    {"text": "I'm getting a 502 error when I try to access my dashboard.", "label": "technical", "difficulty": "easy"},
    {"text": "The search feature is returning no results even though I know the item exists.", "label": "technical", "difficulty": "easy"},
    {"text": "Integration with Slack stopped working after the last update.", "label": "technical", "difficulty": "easy"},
    {"text": "Images are not loading on the product pages.", "label": "technical", "difficulty": "easy"},
    {"text": "The export to CSV button does nothing when I click it.", "label": "technical", "difficulty": "easy"},
    {"text": "My notifications are delayed by several hours.", "label": "technical", "difficulty": "ambiguous"},
    {"text": "The mobile app keeps logging me out after every restart.", "label": "technical", "difficulty": "ambiguous"},
    {"text": "File uploads fail when the file is larger than 10MB.", "label": "technical", "difficulty": "easy"},
    {"text": "The dark mode toggle is not saving my preference.", "label": "technical", "difficulty": "easy"},

    # ── account (easy) ──
    {"text": "I forgot my password and the reset email never arrives.", "label": "account", "difficulty": "easy"},
    {"text": "How do I change the email address on my account?", "label": "account", "difficulty": "easy"},
    {"text": "I want to add a second user to my team account.", "label": "account", "difficulty": "easy"},
    {"text": "My account was locked after too many login attempts.", "label": "account", "difficulty": "easy"},
    {"text": "Can I merge my two accounts into one?", "label": "account", "difficulty": "easy"},
    {"text": "How do I enable two-factor authentication?", "label": "account", "difficulty": "easy"},
    {"text": "I need to update my billing address in my profile.", "label": "account", "difficulty": "ambiguous"},
    {"text": "Please delete my account and all associated data.", "label": "account", "difficulty": "easy"},
    {"text": "I can't log in with my Google SSO anymore.", "label": "account", "difficulty": "ambiguous"},
    {"text": "What permissions does the admin role have?", "label": "account", "difficulty": "easy"},

    # ── shipping (easy) ──
    {"text": "My order has been stuck on 'label created' for five days.", "label": "shipping", "difficulty": "easy"},
    {"text": "Can I change the delivery address after placing the order?", "label": "shipping", "difficulty": "easy"},
    {"text": "I received the wrong item in my package.", "label": "shipping", "difficulty": "easy"},
    {"text": "How long does standard shipping take to Europe?", "label": "shipping", "difficulty": "easy"},
    {"text": "I want to return a product. What is the process?", "label": "shipping", "difficulty": "easy"},
    {"text": "The tracking page shows delivered but I didn't receive anything.", "label": "shipping", "difficulty": "easy"},
    {"text": "Do you offer express shipping for international orders?", "label": "shipping", "difficulty": "easy"},
    {"text": "My package arrived damaged. I need a replacement.", "label": "shipping", "difficulty": "ambiguous"},
    {"text": "Can I pick up the order from your warehouse instead?", "label": "shipping", "difficulty": "easy"},
    {"text": "I need to cancel a shipment that has not been dispatched yet.", "label": "shipping", "difficulty": "easy"},

    # ── general ──
    {"text": "I love the new dashboard design, great work!", "label": "general", "difficulty": "easy"},
    {"text": "Would you consider adding a dark mode?", "label": "general", "difficulty": "easy"},
    {"text": "What are your business hours?", "label": "general", "difficulty": "easy"},
    {"text": "Can you tell me more about your enterprise plan?", "label": "general", "difficulty": "ambiguous"},
    {"text": "I'd like to provide feedback on my recent support experience.", "label": "general", "difficulty": "easy"},
    {"text": "Do you have an API we can use?", "label": "general", "difficulty": "ambiguous"},
    {"text": "Is there a community forum or Slack group?", "label": "general", "difficulty": "easy"},
    {"text": "I have a partnership proposal for your team.", "label": "general", "difficulty": "easy"},
    {"text": "How does your product compare to CompetitorX?", "label": "general", "difficulty": "easy"},
    {"text": "Are you hiring for any engineering positions?", "label": "general", "difficulty": "easy"},
]

df = pd.DataFrame(examples)
print(f"Total examples: {len(df)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts().to_string())
print(f"\nDifficulty distribution:")
print(df["difficulty"].value_counts().to_string())

## 4. Train / Test Split

### Split Rules for a Fair Comparison

- **Test set** — used by ALL three approaches. Never appears in few-shot prompts or fine-tuning data.
- **Few-shot pool** — a small subset of training data used to build few-shot prompts. Must be separate from the test set.
- **Fine-tuning train set** — the full training set (minus the test set). Fine-tuning gets more data than few-shot by design — that is part of what we are measuring.

```
Full dataset (50 examples)
    │
    ├── Test set (30%)  ──── used for ALL evaluations
    │
    └── Train set (70%)
          │
          ├── Few-shot pool (1-2 per class from train) ── used in few-shot prompts
          │
          └── Full train set ── used for fine-tuning
```

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Build a few-shot pool: 2 examples per class from the training set
few_shot_pool = train_df.groupby("label").apply(
    lambda g: g.sample(n=min(2, len(g)), random_state=SEED), include_groups=False
).reset_index(drop=True)

print(f"Train set:      {len(train_df)} examples")
print(f"Test set:       {len(test_df)} examples")
print(f"Few-shot pool:  {len(few_shot_pool)} examples")

print(f"\nTest label distribution:")
print(test_df["label"].value_counts().sort_index().to_string())

# Verify no leakage
test_texts = set(test_df["text"])
few_shot_texts = set(few_shot_pool["text"])
assert len(test_texts & few_shot_texts) == 0, "LEAKAGE: few-shot examples found in test set!"
print("\nNo leakage between few-shot pool and test set.")

## 5. Shared Evaluation Framework

All three approaches are scored with the **same function**. This eliminates evaluation bias.

### Metrics

| Metric | Why |
|---|---|
| **Accuracy** | Overall correctness |
| **Macro F1** | Balanced performance across all classes (important when classes are equal-sized) |
| **Per-class F1** | Identifies which classes are easy/hard for each approach |
| **Accuracy on ambiguous slice** | Shows where the approach struggles on overlapping intents |

### Output Normalization

LLMs sometimes return the label wrapped in quotes, with extra whitespace, or with capital letters. We normalize before scoring so that formatting differences do not count as errors.

In [ ]:
def normalize_label(raw: str) -> str:
    cleaned = raw.strip().lower().strip("\"'`.,!").strip()
    # Handle cases where the model returns extra text around the label
    for label in LABELS:
        if label in cleaned:
            return label
    return cleaned


def evaluate(predictions, references, approach_name, test_frame):
    pred_normalized = [normalize_label(p) for p in predictions]
    ref_list = list(references)

    acc = accuracy_score(ref_list, pred_normalized)
    f1_macro = f1_score(ref_list, pred_normalized, average="macro", zero_division=0)
    report = classification_report(ref_list, pred_normalized, zero_division=0, output_dict=True)

    # Ambiguous slice
    ambig_mask = test_frame["difficulty"] == "ambiguous"
    if ambig_mask.any():
        ambig_pred = [p for p, m in zip(pred_normalized, ambig_mask) if m]
        ambig_ref = [r for r, m in zip(ref_list, ambig_mask) if m]
        ambig_acc = accuracy_score(ambig_ref, ambig_pred)
    else:
        ambig_acc = float("nan")

    # Invalid predictions (not in LABELS)
    invalid = [p for p in pred_normalized if p not in LABELS]

    result = {
        "approach": approach_name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "ambiguous_accuracy": ambig_acc,
        "invalid_predictions": len(invalid),
        "per_class": {
            label: {
                "precision": report.get(label, {}).get("precision", 0),
                "recall": report.get(label, {}).get("recall", 0),
                "f1": report.get(label, {}).get("f1-score", 0),
            }
            for label in LABELS
        },
    }

    print(f"\n{'=' * 55}")
    print(f"  {approach_name}")
    print(f"{'=' * 55}")
    print(f"  Accuracy:            {acc:.1%}")
    print(f"  Macro F1:            {f1_macro:.3f}")
    print(f"  Ambiguous accuracy:  {ambig_acc:.1%}")
    print(f"  Invalid predictions: {len(invalid)}")
    print(f"\n  {'Class':<12} {'Prec':>6} {'Recall':>7} {'F1':>6}")
    print(f"  {'-'*12} {'-'*6} {'-'*7} {'-'*6}")
    for label in LABELS:
        pc = result["per_class"][label]
        print(f"  {label:<12} {pc['precision']:>5.0%} {pc['recall']:>6.0%} {pc['f1']:>5.0%}")

    return result


print("Evaluation framework ready.")

## 6. Approach 1 — Zero-Shot Prompting

The model receives only:

- the list of valid labels with descriptions
- the customer message
- an instruction to return exactly one label

**No examples.** This tests the model's pre-trained understanding of the task.

### Prompt Design

The prompt is deliberately simple and unoptimized. In a real comparison you would iterate on prompt engineering, but starting simple establishes a fair baseline.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.0,
    num_predict=20,
)

ZERO_SHOT_SYSTEM = f'''You are a customer support intent classifier.
Given a customer message, classify it into exactly ONE of these categories:

{chr(10).join(f"- {label}: {desc}" for label, desc in LABEL_DESCRIPTIONS.items())}

Rules:
- Reply with ONLY the category name, nothing else.
- Do not explain your reasoning.
- If uncertain, pick the single best match.'''

print("Zero-shot system prompt:")
print(ZERO_SHOT_SYSTEM)

In [ ]:
def classify_zero_shot(text):
    messages = [
        SystemMessage(content=ZERO_SHOT_SYSTEM),
        HumanMessage(content=f"Customer message: {text}"),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


if RUN_PROMPTING:
    zero_shot_preds = []
    for i, row in test_df.iterrows():
        pred = classify_zero_shot(row["text"])
        zero_shot_preds.append(pred)
        if (i + 1) % 5 == 0:
            print(f"  Zero-shot: {i + 1}/{len(test_df)} done")

    zero_shot_results = evaluate(zero_shot_preds, test_df["label"], "Zero-Shot Prompting", test_df)
else:
    print("Prompting skipped. Set RUN_PROMPTING = True.")
    zero_shot_preds = []
    zero_shot_results = None

## 7. Approach 2 — Few-Shot Prompting

Same model, same system prompt, but now we include **2 labeled examples per class** (10 total) in the prompt.

### Why Few-Shot?

Few-shot prompting often closes a large part of the gap between zero-shot and fine-tuning. If few-shot solves the problem, fine-tuning may not be worth the maintenance cost.

### Few-Shot Construction Rules

- examples come from the **training set only** (never the test set)
- include one "easy" and one "harder" example per class when possible
- keep the same format for every example

In [ ]:
def build_few_shot_block(pool_df):
    lines = ["Here are examples of correctly classified messages:\n"]
    for _, row in pool_df.iterrows():
        lines.append(f'Message: "{row["text"]}"')
        lines.append(f"Category: {row['label']}\n")
    return "\n".join(lines)


FEW_SHOT_BLOCK = build_few_shot_block(few_shot_pool)

FEW_SHOT_SYSTEM = f'''{ZERO_SHOT_SYSTEM}

{FEW_SHOT_BLOCK}'''

print(f"Few-shot system prompt ({len(FEW_SHOT_SYSTEM)} chars):")
print(FEW_SHOT_SYSTEM[:1500])
if len(FEW_SHOT_SYSTEM) > 1500:
    print(f"... [{len(FEW_SHOT_SYSTEM) - 1500} more chars]")

In [ ]:
def classify_few_shot(text):
    messages = [
        SystemMessage(content=FEW_SHOT_SYSTEM),
        HumanMessage(content=f"Customer message: {text}"),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


if RUN_PROMPTING:
    few_shot_preds = []
    for i, row in test_df.iterrows():
        pred = classify_few_shot(row["text"])
        few_shot_preds.append(pred)
        if (i + 1) % 5 == 0:
            print(f"  Few-shot: {i + 1}/{len(test_df)} done")

    few_shot_results = evaluate(few_shot_preds, test_df["label"], "Few-Shot Prompting", test_df)
else:
    print("Prompting skipped. Set RUN_PROMPTING = True.")
    few_shot_preds = []
    few_shot_results = None

## 8. Approach 3 — LoRA Fine-Tuning

### What Changes

Instead of adding examples to the prompt, we **update adapter weights** on the full training set.

### Trade-offs

| Dimension | Prompting | Fine-tuning |
|---|---|---|
| Setup cost | Minutes | Hours to days |
| Per-query cost | Higher (longer prompts with examples) | Lower (short prompt, learned behavior) |
| Maintenance | Change the prompt | Retrain when data changes |
| Consistency | Can drift with model updates | Locked into your adapter |
| Data required | 0–10 examples | 50+ labeled examples |

### Training Data Format

Each training example is a chat turn:
- **system**: classification instructions (same labels as prompting, to keep the comparison fair)
- **user**: the customer message
- **assistant**: the correct label

In [ ]:
FT_SYSTEM_PROMPT = f'''Classify the customer message into exactly one category.
Valid categories: {", ".join(LABELS)}
Reply with only the category name.'''


def to_chat_record(row):
    return {
        "text_input": row["text"],
        "label": row["label"],
        "messages": [
            {"role": "system", "content": FT_SYSTEM_PROMPT},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": row["label"]},
        ],
    }


train_chat = [to_chat_record(row) for _, row in train_df.iterrows()]
print(f"Fine-tuning examples: {len(train_chat)}")
print(f"\nSample formatted record:")
print(json.dumps(train_chat[0]["messages"], indent=2))

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

tokenizer = AutoTokenizer.from_pretrained(BASE_HF_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


ft_train_dataset = Dataset.from_list(train_chat)
ft_train_dataset = ft_train_dataset.map(lambda row: {"text": render_text(row["messages"])})

# Token lengths
lengths = [len(tokenizer(ex["text"]).input_ids) for ex in ft_train_dataset]
print(f"Token length stats:  mean={np.mean(lengths):.0f}  max={max(lengths)}  p95={np.percentile(lengths, 95):.0f}")

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "ft-classification-lora"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=SEED,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_HF_MODEL,
    trust_remote_code=True,
    torch_dtype=(
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    ),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ft_train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainer ready — {len(ft_train_dataset)} examples, {training_args.num_train_epochs} epochs")

In [ ]:
if RUN_FINE_TUNING:
    result = trainer.train()
    trainer.save_model(str(ARTIFACT_DIR / "ft-classification-lora"))
    tokenizer.save_pretrained(str(ARTIFACT_DIR / "ft-classification-lora"))
    print(result)
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNING = True to run.")

## 9. Evaluate the Fine-Tuned Model

After training, run the fine-tuned model on the **same test set** using the same evaluation function.

The fine-tuned model gets a **shorter prompt** — no few-shot examples, no label descriptions. The knowledge is in the weights.

In [ ]:
from peft import PeftModel
from transformers import pipeline as hf_pipeline


def classify_fine_tuned(generator, text):
    messages = [
        {"role": "system", "content": FT_SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = generator(prompt, max_new_tokens=10, do_sample=False, temperature=None)
    generated = output[0]["generated_text"]
    return generated[len(prompt):].strip()


if RUN_FT_EVAL:
    ft_base = AutoModelForCausalLM.from_pretrained(
        BASE_HF_MODEL, trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    ft_model = PeftModel.from_pretrained(ft_base, str(ARTIFACT_DIR / "ft-classification-lora"))
    ft_gen = hf_pipeline("text-generation", model=ft_model, tokenizer=tokenizer)

    ft_preds = []
    for i, row in test_df.iterrows():
        pred = classify_fine_tuned(ft_gen, row["text"])
        ft_preds.append(pred)
        if (i + 1) % 5 == 0:
            print(f"  Fine-tuned: {i + 1}/{len(test_df)} done")

    ft_results = evaluate(ft_preds, test_df["label"], "Fine-Tuned (LoRA)", test_df)
else:
    print("Fine-tuned eval skipped. Set RUN_FT_EVAL = True after training.")
    ft_preds = []
    ft_results = None

## 10. Head-to-Head Comparison

### How to Read the Results

- **Accuracy and F1** — the primary metrics. If zero-shot is already above 90%, the ceiling for improvement is small.
- **Ambiguous accuracy** — the most diagnostic metric. This is where approaches diverge most.
- **Invalid predictions** — labels outside the taxonomy. Fine-tuning almost always eliminates these.
- **Per-class breakdown** — reveals if one approach is systematically weaker on specific intents.

In [ ]:
available_results = []
if zero_shot_results:
    available_results.append(zero_shot_results)
if few_shot_results:
    available_results.append(few_shot_results)
if ft_results:
    available_results.append(ft_results)

if available_results:
    summary = pd.DataFrame([
        {
            "Approach": r["approach"],
            "Accuracy": f"{r['accuracy']:.1%}",
            "Macro F1": f"{r['f1_macro']:.3f}",
            "Ambiguous Acc": f"{r['ambiguous_accuracy']:.1%}",
            "Invalid Preds": r["invalid_predictions"],
        }
        for r in available_results
    ])
    print("HEAD-TO-HEAD COMPARISON")
    print("=" * 70)
    print(summary.to_string(index=False))

    # Per-class F1 comparison
    print(f"\n\nPER-CLASS F1")
    print("=" * 70)
    header = f"  {'Class':<12}" + "".join(f" {r['approach']:>20}" for r in available_results)
    print(header)
    print("  " + "-" * (12 + 21 * len(available_results)))
    for label in LABELS:
        row_str = f"  {label:<12}"
        for r in available_results:
            f1 = r["per_class"][label]["f1"]
            row_str += f" {f1:>19.0%}"
        print(row_str)
else:
    print("No results available yet. Run at least the prompting experiments.")

## 11. Error Analysis

Numbers alone do not tell you what to do next. Look at the **specific examples each approach got wrong** to understand the failure modes.

### What to Look For

- Do all approaches fail on the same examples? → The examples might be genuinely ambiguous or mislabeled.
- Does zero-shot fail but few-shot succeeds? → The model needed calibration examples.
- Does few-shot fail but fine-tuning succeeds? → The model needed weight-level pattern learning.
- Does fine-tuning fail on something prompting got right? → Possible regression from overfitting.

In [ ]:
def error_analysis(predictions, references, test_frame, approach_name):
    pred_normalized = [normalize_label(p) for p in predictions]
    errors = []
    for pred, ref, (_, row) in zip(pred_normalized, references, test_frame.iterrows()):
        if pred != ref:
            errors.append({
                "text": row["text"][:80],
                "true": ref,
                "predicted": pred,
                "difficulty": row["difficulty"],
            })

    if not errors:
        print(f"\n{approach_name}: No errors!")
        return pd.DataFrame()

    error_df = pd.DataFrame(errors)
    print(f"\n{approach_name} — {len(errors)} errors:")
    print(error_df.to_string(index=False))
    return error_df


if zero_shot_preds:
    zs_errors = error_analysis(zero_shot_preds, test_df["label"], test_df, "Zero-Shot")

if few_shot_preds:
    fs_errors = error_analysis(few_shot_preds, test_df["label"], test_df, "Few-Shot")

if ft_preds:
    ft_errors = error_analysis(ft_preds, test_df["label"], test_df, "Fine-Tuned")

## 12. Confusion Matrices

In [ ]:
def show_confusion(predictions, references, approach_name):
    pred_normalized = [normalize_label(p) for p in predictions]
    cm = confusion_matrix(list(references), pred_normalized, labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    print(f"\n{approach_name} — Confusion Matrix (rows=true, cols=predicted)")
    print(cm_df.to_string())
    return cm_df


if zero_shot_preds:
    print("=" * 55)
    show_confusion(zero_shot_preds, test_df["label"], "Zero-Shot")
if few_shot_preds:
    print()
    show_confusion(few_shot_preds, test_df["label"], "Few-Shot")
if ft_preds:
    print()
    show_confusion(ft_preds, test_df["label"], "Fine-Tuned")

## 13. Cost and Latency Analysis

Accuracy is not the only dimension. In production, compare:

| Dimension | Zero-shot | Few-shot | Fine-tuned |
|---|---|---|---|
| **Prompt length** | Short (~200 tokens) | Long (~500+ tokens) | Short (~50 tokens) |
| **Per-query latency** | Baseline | Higher (more tokens to process) | Lower (shorter prompt) |
| **Setup time** | Minutes | Minutes | Hours |
| **Maintenance** | Update prompt text | Update prompt + examples | Retrain pipeline |
| **Monthly cost at 10K queries** | Baseline | ~2.5x prompt tokens | Training cost + baseline inference |
| **Consistency across updates** | May break on model updates | May break on model updates | Adapter locks behavior |

In [ ]:
if RUN_PROMPTING:
    zs_prompt_len = len(tokenizer.encode(ZERO_SHOT_SYSTEM))
    fs_prompt_len = len(tokenizer.encode(FEW_SHOT_SYSTEM))
    ft_prompt_len = len(tokenizer.encode(FT_SYSTEM_PROMPT))

    cost_df = pd.DataFrame([
        {"Approach": "Zero-Shot", "System prompt tokens": zs_prompt_len, "Relative prompt cost": "1.0x"},
        {"Approach": "Few-Shot", "System prompt tokens": fs_prompt_len, "Relative prompt cost": f"{fs_prompt_len/zs_prompt_len:.1f}x"},
        {"Approach": "Fine-Tuned", "System prompt tokens": ft_prompt_len, "Relative prompt cost": f"{ft_prompt_len/zs_prompt_len:.1f}x"},
    ])
    print("PROMPT TOKEN COST COMPARISON")
    print(cost_df.to_string(index=False))

    print(f"\nAt 10,000 queries/day:")
    print(f"  Zero-shot:  {zs_prompt_len * 10_000:>12,} system prompt tokens/day")
    print(f"  Few-shot:   {fs_prompt_len * 10_000:>12,} system prompt tokens/day")
    print(f"  Fine-tuned: {ft_prompt_len * 10_000:>12,} system prompt tokens/day")
else:
    print("Token cost comparison requires RUN_PROMPTING = True.")

## 14. Save Experiment Results

In [ ]:
experiment_log = {
    "timestamp": datetime.now().isoformat(),
    "task": "intent_classification",
    "labels": LABELS,
    "dataset_size": len(df),
    "test_size": len(test_df),
    "train_size": len(train_df),
    "few_shot_pool_size": len(few_shot_pool),
    "ollama_model": OLLAMA_MODEL,
    "hf_model": BASE_HF_MODEL,
    "results": {
        "zero_shot": zero_shot_results,
        "few_shot": few_shot_results,
        "fine_tuned": ft_results,
    },
}

log_path = ARTIFACT_DIR / "experiment_log.json"
log_path.write_text(json.dumps(experiment_log, indent=2, default=str), encoding="utf-8")
print(f"Experiment log saved: {log_path}")

## 15. Interpreting the Results

### Decision Tree

```
                    Zero-shot accuracy
                         │
              ┌──── ≥ 90% ────┐
              │                │
     Good enough?        Few-shot improves?
              │                │
         ┌── Yes          ┌── Yes ──┐
         │                │         │
    Stop here.       Few-shot    Fine-tuning
    Use zero-shot.   is enough.  may not be
                                 worth it.
              │                │
         ┌── No           ┌── No ──┐
         │                │         │
    Try few-shot.    Fine-tune.  The problem
                     The model   may need a
                     needs       different
                     weight      approach.
                     updates.
```

### When Each Approach Wins

| Scenario | Winner | Why |
|---|---|---|
| Simple taxonomy, high-quality labels, low volume | Zero-shot | No setup cost, good enough accuracy |
| Moderate complexity, some confusable classes | Few-shot | Examples disambiguate edge cases cheaply |
| High volume, strict consistency required, latency matters | Fine-tuning | Short prompts, stable behavior, lower per-query cost |
| Taxonomy changes frequently | Few-shot | Easier to update examples than retrain |
| Model provider may change models | Fine-tuning | Your adapter preserves tested behavior |
| Very few labeled examples (<20) | Zero-shot or few-shot | Not enough data to fine-tune reliably |

### Common Pitfalls

1. **Comparing fine-tuning against a weak prompt baseline.** Always optimize your prompt first.
2. **Using accuracy alone.** Check per-class F1 and ambiguous accuracy.
3. **Ignoring maintenance cost.** A 3% accuracy gain from fine-tuning may not justify a retraining pipeline.
4. **Leaking test data into few-shot prompts.** Guaranteed to overestimate few-shot performance.
5. **Fine-tuning on noisy labels.** If annotators disagree, the model learns confusion.

## 16. Key Takeaways

### What We Built

- A **controlled comparison** of zero-shot prompting, few-shot prompting, and LoRA fine-tuning on the same intent classification task
- A **shared evaluation framework** with accuracy, macro F1, per-class breakdown, and ambiguous-example analysis
- **Error analysis and confusion matrices** to understand failure modes, not just aggregate numbers
- A **cost comparison** showing prompt token overhead across approaches

### Practical Rules of Thumb

1. **Start with zero-shot.** If it works, stop there. Complexity has a maintenance cost.

2. **Try few-shot before fine-tuning.** Adding 2 examples per class often closes most of the gap.

3. **Fine-tune when consistency at scale matters.** Fine-tuning is not about peak accuracy — it is about reliable, low-latency behavior across thousands of queries.

4. **Always evaluate on ambiguous examples separately.** That is where approaches actually diverge.

5. **Document your experimental setup.** Model name, temperature, prompt text, data split, seed — all of it. Results without methodology are not reproducible.

6. **Re-run the comparison when you change models.** A new base model may change which approach wins.